[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/AM/blob/main/04_Trees.ipynb)


# Introdução ao Aprendizado de Máquina

**Professor: Diogo Ferreira de Lima Silva**  
**TPP - UFF**


# Árvores de Decisão


## 1. Importação dos pacotes

Vamos começar importando os pacotes que serão usados ao longo da aula.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, r2_score

np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True



## 2. Árvores de classificação com o conjunto Iris

Usaremos inicialmente o conhecido conjunto **Iris**, que contém 150 flores, 3 classes e 4 atributos numéricos.

Para facilitar algumas visualizações em 2D, começaremos usando apenas:

- comprimento da pétala;
- largura da pétala.


In [ ]:
iris = load_iris()

X = iris.data[:, 2:4]  # Todas as linhas, mas apenas as colunas de comprimento e largura da pétala
y = iris.target

print("Nomes dos atributos:", iris.feature_names)
print("Classes:", iris.target_names)
print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

pd.DataFrame(X, columns=iris.feature_names[2:4]).head()



### Visualização inicial dos dados

Antes de treinar o modelo, vale observar como as classes se distribuem no espaço dos atributos.


In [ ]:
markers = ["o", "s", "^"]
labels = iris.target_names

for classe, marcador, nome in zip(np.unique(y), markers, labels):
    plt.scatter(
        X[y == classe, 0],
        X[y == classe, 1],
        label=nome,
        marker=marcador,
        s=60,
        alpha=0.8
    )

plt.xlabel("Comprimento da pétala (cm)")
plt.ylabel("Largura da pétala (cm)")
plt.title("Conjunto Iris usando 2 atributos")
plt.legend()
plt.show()



## 3. Treinando uma árvore rasa

Vamos treinar uma árvore com `max_depth=2`. Essa restrição torna a árvore pequena e mais fácil de interpretar.


In [ ]:
tree_clf = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_clf.fit(X, y)

print("Profundidade da árvore:", tree_clf.get_depth())
print("Número de folhas:", tree_clf.get_n_leaves())



### Visualizando a árvore

Cada nó interno contém uma regra do tipo `atributo <= ponto_de_corte`.  
As folhas mostram a distribuição das classes e a classe prevista.


In [ ]:
plt.figure(figsize=(14, 8))
plot_tree(
    tree_clf,
    feature_names=iris.feature_names[2:4],
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=11
)
plt.title("Árvore de classificação (max_depth=2)")
plt.show()



### Regras em formato textual

Essa forma costuma ser útil para reforçar a ideia de que uma árvore representa uma sequência hierárquica de decisões.


In [ ]:
print(export_text(tree_clf, feature_names=iris.feature_names[2:4]))

## 4. Regiões de decisão

Uma árvore particiona o espaço de atributos em regiões. Dentro de cada região, a previsão é constante.


In [ ]:
from matplotlib.colors import ListedColormap

def plot_decision_boundary(clf, X, y, axes=None, title=None):
    if axes is None:
        axes = [X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, X[:, 1].min() - 0.5, X[:, 1].max() + 0.5]

    x1s = np.linspace(axes[0], axes[1], 300)
    x2s = np.linspace(axes[2], axes[3], 300)
    x1, x2 = np.meshgrid(x1s, x2s)
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_new).reshape(x1.shape)

    cmap_bg = ListedColormap(["#fff7bc", "#c7d2fe", "#bbf7d0"])
    plt.contourf(x1, x2, y_pred, alpha=0.45, cmap=cmap_bg)

    markers = ["o", "s", "^"]
    labels = iris.target_names
    for classe, marcador, nome in zip(np.unique(y), markers, labels):
        plt.scatter(X[y == classe, 0], X[y == classe, 1], label=nome, marker=marcador, s=45)

    plt.xlabel("Comprimento da pétala (cm)")
    plt.ylabel("Largura da pétala (cm)")
    plt.xlim(axes[0], axes[1])
    plt.ylim(axes[2], axes[3])
    if title:
        plt.title(title)
    plt.legend()
    plt.show()

plot_decision_boundary(tree_clf, X, y, title="Regiões de decisão da árvore")



## 5. Realizando previsões

Considere uma flor com:

- comprimento da pétala = **5.0 cm**
- largura da pétala = **1.5 cm**

Vamos calcular as probabilidades estimadas e a classe prevista.


In [ ]:
novo_exemplo = np.array([[5.0, 1.5]])

probs = tree_clf.predict_proba(novo_exemplo)
pred = tree_clf.predict(novo_exemplo)

print("Probabilidades por classe:", probs)
print("Classe prevista (índice):", pred[0])
print("Classe prevista (nome):", iris.target_names[pred[0]])


### Exercício 1

Altere o valor de `novo_exemplo` e teste diferentes combinações de comprimento/largura da pétala.

Tente responder:

1. Em quais regiões a classe prevista muda?
2. Há exemplos para os quais a árvore parece “muito confiante”? 
3. Compare o ponto escolhido com a figura das regiões de decisão.


In [ ]:
# Teste seu código aqui

## 6. Avaliando capacidade de generalização

Até aqui treinamos e analisamos a árvore sobre o conjunto completo. Agora vamos fazer uma divisão entre **treino** e **teste**, para avaliar generalização.

**Importante:** Na prática, o conjunto de teste não deve ser usado para escolher a melhor combinação de hiperparâmetros. Isso pode trazer um problema de viés no modelo. É comum o uso de um terceiro grupo, chamado de **conjunto de validação**. Veremos isso adiante na disciplina.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

modelos = {
    "max_depth=1": DecisionTreeClassifier(max_depth=1, random_state=42),
    "max_depth=2": DecisionTreeClassifier(max_depth=2, random_state=42),
    "max_depth=3": DecisionTreeClassifier(max_depth=3, random_state=42),
    "sem limite": DecisionTreeClassifier(random_state=42),
}

resultados = []
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    acc_treino = accuracy_score(y_train, modelo.predict(X_train))
    acc_teste = accuracy_score(y_test, modelo.predict(X_test))
    resultados.append([nome, acc_treino, acc_teste, modelo.get_depth(), modelo.get_n_leaves()])

pd.DataFrame(resultados, columns=["modelo", "acc_treino", "acc_teste", "profundidade", "folhas"])


Observe que aumentar a complexidade da árvore tende a melhorar o ajuste no treino.  

No entanto, isso **não garante** melhor desempenho no teste.




### Exercício 2

Inclua novos modelos alterando, por exemplo:

- `max_depth`
- `min_samples_leaf`
- `min_samples_split`

Depois, compare o efeito sobre:

1. acurácia de treino;
2. acurácia de teste;
3. profundidade final da árvore.



## 7. Exemplo visual de overfitting

Para enxergar melhor o risco de overfitting, vamos usar um conjunto de dados sintético mais desafiador.


In [ ]:

X_moons, y_moons = make_moons(n_samples=250, noise=0.30, random_state=42)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons, y_moons, test_size=0.30, random_state=42
)

clf_shallow = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_deep = DecisionTreeClassifier(random_state=42)

clf_shallow.fit(X_train_m, y_train_m)
clf_deep.fit(X_train_m, y_train_m)

print("Acurácia treino - árvore rasa:", accuracy_score(y_train_m, clf_shallow.predict(X_train_m)))
print("Acurácia teste  - árvore rasa:", accuracy_score(y_test_m, clf_shallow.predict(X_test_m)))
print()
print("Acurácia treino - árvore profunda:", accuracy_score(y_train_m, clf_deep.predict(X_train_m)))
print("Acurácia teste  - árvore profunda:", accuracy_score(y_test_m, clf_deep.predict(X_test_m)))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, clf, titulo in zip(
    axes,
    [clf_shallow, clf_deep],
    ["Árvore rasa", "Árvore profunda"]
):
    plt.sca(ax)
    x1s = np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 300)
    x2s = np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 300)
    x1, x2 = np.meshgrid(x1s, x2s)
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_new).reshape(x1.shape)

    plt.contourf(x1, x2, y_pred, alpha=0.35)
    plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, edgecolor="k", s=30)
    plt.title(titulo)
    plt.xlabel("x1")
    plt.ylabel("x2")

plt.suptitle("Comparação visual: árvore simples vs árvore complexa")
plt.tight_layout()
plt.show()



### Exercício 3

Repita o experimento acima trocando o parâmetro da árvore profunda por algum mecanismo de regularização, por exemplo:

- `min_samples_leaf=5`
- `max_depth=4`
- `min_samples_split=10`

Discuta: a fronteira ficou mais suave? O desempenho em teste melhorou?



## 8. Importância dos atributos

Árvores também podem ser usadas para obter uma noção da relevância relativa dos atributos.

Agora vamos usar os **4 atributos** do conjunto Iris.


In [ ]:
X_full = iris.data
y_full = iris.target

full_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
full_tree.fit(X_full, y_full)

importancias = pd.DataFrame({
    "atributo": iris.feature_names,
    "importancia": full_tree.feature_importances_
}).sort_values("importancia", ascending=False)

importancias


In [ ]:
plt.bar(importancias["atributo"], importancias["importancia"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importância")
plt.title("Importância dos atributos na árvore")
plt.show()



### Exercício 4

Treine uma nova árvore usando os 4 atributos e altere o `max_depth`.

Depois:

1. compare as importâncias obtidas;
2. verifique se o atributo mais importante permanece o mesmo;
3. interprete esse resultado à luz das divisões feitas pela árvore.



## 9. Matriz de confusão

A matriz de confusão ajuda a identificar **quais classes** estão sendo confundidas entre si.


In [ ]:

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y_full, test_size=0.30, stratify=y_full, random_state=42
)

clf_cm = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_cm.fit(X_train_f, y_train_f)
y_pred_f = clf_cm.predict(X_test_f)

cm = confusion_matrix(y_test_f, y_pred_f)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names).plot(cmap="Blues")
plt.title("Matriz de confusão - árvore de classificação")
plt.show()



## 10. Árvores de regressão

Agora vamos passar para um problema de regressão.  
A ideia é mostrar que a árvore aprende uma função **por partes**, isto é, constante em intervalos.


In [ ]:

np.random.seed(42)
m = 200
X_reg = np.random.rand(m, 1)
y_reg = 4 * (X_reg - 0.5) ** 2 + np.random.randn(m, 1) / 10

pd.DataFrame


In [ ]:

def plot_regression_predictions(model, X, y, axes=[0, 1, -0.2, 1.1], title=None):
    x_new = np.linspace(axes[0], axes[1], 500).reshape(-1, 1)
    y_pred = model.predict(x_new)

    plt.plot(X, y, "b.", alpha=0.7)
    plt.plot(x_new, y_pred, "r.-", linewidth=2)
    plt.axis(axes)
    plt.xlabel("x")
    plt.ylabel("y")
    if title:
        plt.title(title)

reg_depth2 = DecisionTreeRegressor(max_depth=2, random_state=42)
reg_depth5 = DecisionTreeRegressor(max_depth=5, random_state=42)

reg_depth2.fit(X_reg, y_reg)
reg_depth5.fit(X_reg, y_reg)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plot_regression_predictions(reg_depth2, X_reg, y_reg, title="Árvore de regressão (max_depth=2)")

plt.subplot(1, 2, 2)
plot_regression_predictions(reg_depth5, X_reg, y_reg, title="Árvore de regressão (max_depth=5)")

plt.tight_layout()
plt.show()



### Comentário didático

Quanto mais profunda a árvore de regressão, mais “recortada” tende a ficar a função estimada.  
Isso novamente abre espaço para discutir o compromisso entre:

- ajuste aos dados de treino;
- simplicidade do modelo;
- capacidade de generalização.



## 11. Regularização em regressão

Vamos comparar uma árvore sem restrições com outra mais regularizada.


In [ ]:

reg_sem_restricao = DecisionTreeRegressor(random_state=42)
reg_regularizada = DecisionTreeRegressor(random_state=42, min_samples_leaf=10)

reg_sem_restricao.fit(X_reg, y_reg)
reg_regularizada.fit(X_reg, y_reg)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plot_regression_predictions(reg_sem_restricao, X_reg, y_reg, title="Sem restrições")

plt.subplot(1, 2, 2)
plot_regression_predictions(reg_regularizada, X_reg, y_reg, title="Com min_samples_leaf=10")

plt.tight_layout()
plt.show()



### Exercício 5

Teste outros valores para `min_samples_leaf`, por exemplo:

- 2
- 5
- 20

Depois, descreva como muda:

1. a suavidade da curva prevista;
2. o risco de overfitting;
3. a interpretabilidade do modelo.



## 12. Avaliação quantitativa em regressão

Para completar, vamos separar treino e teste e comparar o desempenho de alguns modelos com métricas usuais de regressão.


In [ ]:

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.30, random_state=42)

modelos_reg = {
    "max_depth=2": DecisionTreeRegressor(max_depth=2, random_state=42),
    "max_depth=5": DecisionTreeRegressor(max_depth=5, random_state=42),
    "sem limite": DecisionTreeRegressor(random_state=42),
    "min_samples_leaf=10": DecisionTreeRegressor(min_samples_leaf=10, random_state=42),
}

linhas = []
for nome, modelo in modelos_reg.items():
    modelo.fit(X_train_r, y_train_r)
    pred_train = modelo.predict(X_train_r)
    pred_test = modelo.predict(X_test_r)
    linhas.append([
        nome,
        mean_squared_error(y_train_r, pred_train),
        mean_squared_error(y_test_r, pred_test),
        r2_score(y_train_r, pred_train),
        r2_score(y_test_r, pred_test),
        modelo.get_depth(),
        modelo.get_n_leaves(),
    ])

pd.DataFrame(
    linhas,
    columns=["modelo", "MSE_treino", "MSE_teste", "R2_treino", "R2_teste", "profundidade", "folhas"]
)
